# Geometric EEG SSL v2 — Cross-Montage Pretrain Notebook (GPU)

**Purpose:** Pretrain v2 with mixed cross-montage corpus, three
leave-one-dataset-out runs. Headline experiment for v2.

**Prereq:** Run `colab_download.ipynb` first to cache all three datasets
(PhysioNet MI, BCIC-2B, Sleep-EDFx) and their preprocessed signal arrays.
v2 reuses v1's signal cache unchanged — only `ch_pos` is recomputed
on the fly via `src/v2/preprocess.ch_pos_from_names` (fixed-scale,
shared across montages).

Each pretrain cell auto-resumes from the latest checkpoint if interrupted.


## 1. Install dependencies

In [ ]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy


## 2. Mount Google Drive + paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data dir (raw downloads from colab_download.ipynb)
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.environ['MNE_DATA'] = MNE_DATA_DIR

# v1 signal cache; reused unchanged by v2.
CACHE_ROOT = f'{DRIVE_ROOT}/cache'
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
if not os.path.isdir(CACHE_ROOT):
    print(f'WARNING: no cache at {CACHE_ROOT}. Loaders will reprocess; '
          'run colab_download.ipynb section 5 to build the cache once.')
else:
    print(f'Reusing v1 signal cache at {CACHE_ROOT}')

# v2 checkpoints go to a dedicated subtree so they don't shadow v1's.
CKPT_ROOT = f'{DRIVE_ROOT}/runs/pretrain'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Checkpoints -> {CKPT_ROOT}')


## 2b. Keep Colab alive

Colab sessions die when the browser tab loses focus or the laptop sleeps.
Checkpoints saved every 10 epochs keep work recoverable, but the keep-alive
ping below buys uninterrupted long runs. Re-run after any browser refresh.


In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect() {
  const btn = document.querySelector("colab-connect-button");
  if (btn && btn.shadowRoot) {
    const inner = btn.shadowRoot.querySelector("#connect");
    if (inner) inner.click();
  }
  console.log("colab keep-alive ping " + new Date().toLocaleTimeString());
}
if (window._colabKeepAlive) clearInterval(window._colabKeepAlive);
window._colabKeepAlive = setInterval(ClickConnect, 60000);
console.log("colab keep-alive armed (60s interval)");
'''))
print('Keep-alive armed.')


## 3. Clone repo (v2-improvements branch)

In [ ]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone -b v2-improvements https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} checkout v2-improvements && git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR, '(branch v2-improvements)')


## 4. Verify the v2 stack imports + smoke

Quick sanity check the v2 modules import cleanly. If this fails, every
pretrain cell below will too -- fix imports first.


In [ ]:
import subprocess
out = subprocess.run(
    ['python', f'{REPO_DIR}/tests/smoke_test_v2.py'],
    cwd=REPO_DIR, capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)
print(out.stdout)
if out.returncode != 0:
    print('STDERR:', out.stderr)
    raise RuntimeError('v2 smoke test failed')


## 5. Verify all three datasets are cached

v2 needs all three for the three leave-one-out splits. If any is missing,
run the corresponding section of `colab_download.ipynb`.


In [ ]:
import os, glob

# v2 reuses v1's preprocessing cache. The pretrain script never needs the
# raw MNE downloads if the cache fingerprints match -- so we accept either.
raw_paths = {
    'PhysioNet MI': os.path.join(MNE_DATA_DIR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0'),
    'BCIC-2B':      os.path.join(MNE_DATA_DIR, 'MNE-bnci-data'),
    'Sleep-EDFx':   os.path.join(MNE_DATA_DIR, 'physionet-sleep-data'),
}
cache_prefixes = {
    'PhysioNet MI': 'physionet_mi_pretrain_',
    'BCIC-2B':      'bcic_2b_pretrain_',
    'Sleep-EDFx':   'sleep_edfx_pretrain_',
}

cache_files = set(os.listdir(CACHE_ROOT)) if os.path.isdir(CACHE_ROOT) else set()

missing = []
for name in raw_paths:
    has_raw = os.path.isdir(raw_paths[name])
    has_cache = any(f.startswith(cache_prefixes[name]) for f in cache_files)
    status = []
    if has_cache:
        status.append('cache')
    if has_raw:
        status.append('raw')
    if not status:
        status = ['MISSING']
        missing.append(name)
    print(f'  {name:14s}  [{" + ".join(status)}]')

if missing:
    raise RuntimeError(
        f'Missing both cache and raw for: {missing}. '
        'Run colab_download.ipynb sections 4 (download) and 5 (build cache).'
    )
print('All three datasets have either a cache or raw download present.')


## 6. Per-dataset budget

The three datasets have very different epoch counts:

| Dataset | Epochs available | Channels |
|---|---|---|
| PhysioNet MI | ~9,450 | 64 |
| BCIC-2B | ~6,500 | 3 |
| Sleep-EDFx | ~1,000,000 | 2 (bipolar) |

The v2.2 claim is "distribution of g_ij matters, count doesn't" -- so we cap
each dataset at the same budget per pass. Small datasets are oversampled
with replacement; Sleep-EDFx is subsampled hard.

`EPOCHS_PER_DATASET = 8000` gives roughly the same wall-clock per pass as v1
single-dataset pretraining (which ran ~9,450 PhysioNet epochs/epoch). Bump
or lower as compute allows.


In [ ]:
EPOCHS_PER_DATASET = 8000
print(f'budget per dataset per pass = {EPOCHS_PER_DATASET}')


## 7. Pretrain v2 -- `no_sleep` (Sleep-EDFx held out)

Mixed corpus: `physionet_mi,bcic_2b`. Held out: the third dataset, for downstream
zero-shot eval. Checkpoints saved every 10 epochs to Drive.


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_no_sleep'
os.makedirs(CKPT_DIR, exist_ok=True)

_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_no_sleep: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --pretrain-datasets physionet_mi,bcic_2b \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a {CKPT_DIR}/train_log.txt


## 7. Pretrain v2 -- `no_bcic` (BCIC-2B held out)

Mixed corpus: `physionet_mi,sleep_edfx`. Held out: the third dataset, for downstream
zero-shot eval. Checkpoints saved every 10 epochs to Drive.


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_no_bcic'
os.makedirs(CKPT_DIR, exist_ok=True)

_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_no_bcic: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --pretrain-datasets physionet_mi,sleep_edfx \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a {CKPT_DIR}/train_log.txt


## 7. Pretrain v2 -- `no_phys` (PhysioNet MI held out)

Mixed corpus: `bcic_2b,sleep_edfx`. Held out: the third dataset, for downstream
zero-shot eval. Checkpoints saved every 10 epochs to Drive.


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_no_phys'
os.makedirs(CKPT_DIR, exist_ok=True)

_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_no_phys: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --pretrain-datasets bcic_2b,sleep_edfx \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a {CKPT_DIR}/train_log.txt


## 8. Checkpoint inventory

In [ ]:
import glob, os

for tag in ['no_sleep', 'no_bcic', 'no_phys']:
    d = f'{CKPT_ROOT}/v2_{tag}'
    ckpts = sorted(glob.glob(f'{d}/epoch_*.pt'))
    last = ckpts[-1].split('/')[-1] if ckpts else 'NONE'
    spec = os.path.join(d, 'v2_run.txt')
    spec_str = open(spec).read().strip() if os.path.exists(spec) else '(no v2_run.txt)'
    print(f'v2_{tag:9s}  ckpts={len(ckpts):3d}  latest={last}')
    print('  ' + spec_str.replace('\n', '\n  '))
    print()


## Done

Three v2 pretrain runs complete. Each `runs/pretrain/v2_<tag>/v2_run.txt`
records which dataset is held out for downstream zero-shot eval.

Next: switch to the v2 experiment notebook (to be written) for probe
evaluation on the held-out dataset per run.
